Credits: Forked from [deep-learning-keras-tensorflow](https://github.com/leriomaggio/deep-learning-keras-tensorflow) by Valerio Maggio

# Convolution Nets for MNIST

Deep Learning models can take quite a bit of time to run, particularly if GPU isn't used. 

In the interest of time, you could sample a subset of observations (e.g. $1000$) that are a particular number of your choice (e.g. $6$) and $1000$ observations that aren't that particular number (i.e. $\neq 6$). 

We will build a model using that and see how it performs on the test dataset

In [ ]:
#Import the required libraries
import numpy as np
np.random.seed(1338)

from keras.datasets import mnist

Import the libraries needed for this section:
- **keras**
- **Sequential**

In [ ]:
from keras.models import Sequential

Import the libraries needed for this section:
- **keras**
- **Dense,**

In [ ]:
from keras.layers.core import Dense, Dropout, Activation, Flatten

Import the libraries needed for this section:
- **keras**
- **Convolution2D**
- **MaxPooling2D**

In [ ]:
from keras.layers.convolutional import Convolution2D
from keras.layers.pooling import MaxPooling2D

Import the libraries needed for this section:
- **keras**
- **np_utils**
- **SGD**

In [ ]:
from keras.utils import np_utils
from keras.optimizers import SGD

## Loading Data

### Load the training and testing data

The code below implements this step in the AI/ML workflow.

In [ ]:
path_to_dataset = "euroscipy_2016_dl-keras/data/mnist.pkl.gz"

#Load the training and testing data
(X_train, y_train), (X_test, y_test) = mnist.load_data(path_to_dataset)

The code below implements the next step in this deep learning workflow. See the inline comments for details on each operation.

In [ ]:
X_test_orig = X_test

## Data Preparation

The code below implements the next step in this deep learning workflow. See the inline comments for details on each operation.

In [ ]:
img_rows, img_cols = 28, 28

X_train = X_train.reshape(X_train.shape[0], 1, img_rows, img_cols)
X_test = X_test.reshape(X_test.shape[0], 1, img_rows, img_cols)

X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

X_train /= 255
X_test /= 255

**Random Seed:** Setting a seed ensures **reproducibility** — the same random numbers are generated each time the code runs. This is critical in ML experiments because:
- Train/test splits will be the same
- Weight initialization will be identical
- Any randomized algorithm (dropout, data augmentation) will behave consistently

Without a fixed seed, your results would vary between runs, making it impossible to debug or compare experiments.

In [ ]:
# Seed for reproducibilty
np.random.seed(1338)

# Test data
X_test = X_test.copy()
Y = y_test.copy()

# Converting the output to binary classification(Six=1,Not Six=0)
Y_test = Y == 6
Y_test = Y_test.astype(int)

# Selecting the 5918 examples where the output is 6
X_six = X_train[y_train == 6].copy()
Y_six = y_train[y_train == 6].copy()

# Selecting the examples where the output is not 6
X_not_six = X_train[y_train != 6].copy()
Y_not_six = y_train[y_train != 6].copy()

# Selecting 6000 random examples from the data that 
# only contains the data where the output is not 6
random_rows = np.random.randint(0,X_six.shape[0],6000)
X_not_six = X_not_six[random_rows]
Y_not_six = Y_not_six[random_rows]

Appending the data with output as 6 and data with output as <> 6
Reshaping the appended data to appropraite form
Appending the labels and converting the labels to

The code below implements this step in the deep learning workflow.

In [ ]:
# Appending the data with output as 6 and data with output as <> 6
X_train = np.append(X_six,X_not_six)

# Reshaping the appended data to appropraite form
X_train = X_train.reshape(X_six.shape[0] + X_not_six.shape[0], 
                          1, img_rows, img_cols)

# Appending the labels and converting the labels to 
# binary classification(Six=1,Not Six=0)
Y_labels = np.append(Y_six,Y_not_six)
Y_train = Y_labels == 6 
Y_train = Y_train.astype(int)

Run this cell and inspect the output to verify the deep learning operations produce the expected results.

In [ ]:
print(X_train.shape, Y_labels.shape, X_test.shape, Y_test.shape)

### Converting the classes to its binary categorical form

The code below implements this step in the AI/ML workflow.

In [ ]:
# Converting the classes to its binary categorical form
nb_classes = 2
Y_train = np_utils.to_categorical(Y_train, nb_classes)
Y_test = np_utils.to_categorical(Y_test, nb_classes)

# A simple CNN

### Initializing the values for the convolution neural network

number of convolutional filters to use size of pooling area for max pooling

In [ ]:
#Initializing the values for the convolution neural network
nb_epoch = 2
batch_size = 128
# number of convolutional filters to use
nb_filters = 32
# size of pooling area for max pooling
nb_pool = 2
# convolution kernel size
nb_conv = 3

sgd = SGD(lr=0.1, decay=1e-6, momentum=0.9, nesterov=True)

### Step 1: Model Definition

**Softmax Function:** Converts a vector of raw scores (logits) into a **probability distribution** — all values between 0 and 1 that sum to 1:

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

Softmax amplifies the largest values and suppresses smaller ones. It's used as the final layer in multi-class classification: the output tells you the model's confidence for each class.

**ReLU (Rectified Linear Unit):** The most common activation function in modern neural networks:

$$\text{ReLU}(x) = \max(0, x)$$

It simply passes positive values through and zeroes out negatives. Why it works so well:
- **No vanishing gradient** for positive inputs (gradient is always 1)
- **Sparse activation**: many neurons output 0, making the network efficient
- **Computationally fast**: just a threshold, no exponentials

Variants like Leaky ReLU ($\max(0.01x, x)$) and GELU fix the "dying ReLU" problem where neurons can permanently output 0.

In [ ]:
model = Sequential()

model.add(Convolution2D(nb_filters, nb_conv, nb_conv,
                        border_mode='valid',
                        input_shape=(1, img_rows, img_cols)))
model.add(Activation('relu'))

model.add(Flatten())
model.add(Dense(nb_classes))
model.add(Activation('softmax'))

### Step 2: Compile

The code below implements the next step in this deep learning workflow. See the inline comments for details on each operation.

In [ ]:
model.compile(loss='categorical_crossentropy',
              optimizer='sgd',
              metrics=['accuracy'])

### Step 3: Fit

**Model Training (.fit()):** The `.fit()` method is where the model learns from data. It adjusts the model's internal parameters to minimize prediction errors on the training data.

For different model types, `.fit()` does different things:
- **Linear models**: Finds the best-fit line/plane (minimizes squared error)
- **Decision trees**: Recursively splits data to separate classes/values
- **Neural networks**: Runs gradient descent over many epochs
- **Transformers (StandardScaler, PCA)**: Computes statistics (mean, variance, components) from the training data

In [ ]:
model.fit(X_train, Y_train, batch_size=batch_size, 
          nb_epoch=nb_epoch,verbose=1,
          validation_data=(X_test, Y_test))

### Step 4: Evaluate

Evaluating the model on the test data

The code below implements this step in the deep learning workflow.

In [ ]:
# Evaluating the model on the test data    
score, accuracy = model.evaluate(X_test, Y_test, verbose=0)
print('Test score:', score)
print('Test accuracy:', accuracy)

### Let's plot our model Predictions!

Import the libraries needed for this section:
- **matplotlib**

In [ ]:
import matplotlib.pyplot as plt

%matplotlib inline

**Model Prediction (.predict()):** After training, `.predict()` applies the learned model to new (unseen) data to generate predictions.

- **Classification**: Returns predicted class labels
- **Regression**: Returns predicted continuous values

The quality of predictions depends on how well the model was trained and whether the new data resembles the training distribution.

**Image Display:** Renders a 2D array as an image. Used for:
- Displaying actual images (RGB arrays)
- Visualizing weight matrices, attention maps, or feature maps in neural networks
- Showing heatmaps of correlation matrices

Use `cmap` to set the colormap, `aspect` to control scaling.

In [ ]:
slice = 15
predicted = model.predict(X_test[:slice]).argmax(-1)

plt.figure(figsize=(16,8))
for i in range(slice):
    plt.subplot(1, slice, i+1)
    plt.imshow(X_test_orig[i], interpolation='nearest')
    plt.text(0, 0, predicted[i], color='black', 
             bbox=dict(facecolor='white', alpha=1))
    plt.axis('off')

# Adding more Dense Layers

**Softmax Function:** Converts a vector of raw scores (logits) into a **probability distribution** — all values between 0 and 1 that sum to 1:

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

Softmax amplifies the largest values and suppresses smaller ones. It's used as the final layer in multi-class classification: the output tells you the model's confidence for each class.

**ReLU (Rectified Linear Unit):** The most common activation function in modern neural networks:

$$\text{ReLU}(x) = \max(0, x)$$

It simply passes positive values through and zeroes out negatives. Why it works so well:
- **No vanishing gradient** for positive inputs (gradient is always 1)
- **Sparse activation**: many neurons output 0, making the network efficient
- **Computationally fast**: just a threshold, no exponentials

Variants like Leaky ReLU ($\max(0.01x, x)$) and GELU fix the "dying ReLU" problem where neurons can permanently output 0.

In [ ]:
model = Sequential()
model.add(Convolution2D(nb_filters, nb_conv, nb_conv,
                        border_mode='valid',
                        input_shape=(1, img_rows, img_cols)))
model.add(Activation('relu'))

model.add(Flatten())
model.add(Dense(128))
model.add(Activation('relu'))

model.add(Dense(nb_classes))
model.add(Activation('softmax'))

**Model Training (.fit()):** The `.fit()` method is where the model learns from data. It adjusts the model's internal parameters to minimize prediction errors on the training data.

For different model types, `.fit()` does different things:
- **Linear models**: Finds the best-fit line/plane (minimizes squared error)
- **Decision trees**: Recursively splits data to separate classes/values
- **Neural networks**: Runs gradient descent over many epochs
- **Transformers (StandardScaler, PCA)**: Computes statistics (mean, variance, components) from the training data

In [ ]:
model.compile(loss='categorical_crossentropy',
              optimizer='sgd',
              metrics=['accuracy'])

model.fit(X_train, Y_train, batch_size=batch_size, 
          nb_epoch=nb_epoch,verbose=1,
          validation_data=(X_test, Y_test))

Evaluating the model on the test data

The code below implements this step in the deep learning workflow.

In [ ]:
#Evaluating the model on the test data    
score, accuracy = model.evaluate(X_test, Y_test, verbose=0)
print('Test score:', score)
print('Test accuracy:', accuracy)

# Adding Dropout

**Softmax Function:** Converts a vector of raw scores (logits) into a **probability distribution** — all values between 0 and 1 that sum to 1:

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

Softmax amplifies the largest values and suppresses smaller ones. It's used as the final layer in multi-class classification: the output tells you the model's confidence for each class.

**ReLU (Rectified Linear Unit):** The most common activation function in modern neural networks:

$$\text{ReLU}(x) = \max(0, x)$$

It simply passes positive values through and zeroes out negatives. Why it works so well:
- **No vanishing gradient** for positive inputs (gradient is always 1)
- **Sparse activation**: many neurons output 0, making the network efficient
- **Computationally fast**: just a threshold, no exponentials

Variants like Leaky ReLU ($\max(0.01x, x)$) and GELU fix the "dying ReLU" problem where neurons can permanently output 0.

In [ ]:
model = Sequential()

model.add(Convolution2D(nb_filters, nb_conv, nb_conv,
                        border_mode='valid',
                        input_shape=(1, img_rows, img_cols)))
model.add(Activation('relu'))

model.add(Flatten())
model.add(Dense(128))
model.add(Activation('relu'))
model.add(Dropout(0.5))
model.add(Dense(nb_classes))
model.add(Activation('softmax'))

**Model Training (.fit()):** The `.fit()` method is where the model learns from data. It adjusts the model's internal parameters to minimize prediction errors on the training data.

For different model types, `.fit()` does different things:
- **Linear models**: Finds the best-fit line/plane (minimizes squared error)
- **Decision trees**: Recursively splits data to separate classes/values
- **Neural networks**: Runs gradient descent over many epochs
- **Transformers (StandardScaler, PCA)**: Computes statistics (mean, variance, components) from the training data

In [ ]:
model.compile(loss='categorical_crossentropy',
              optimizer='sgd',
              metrics=['accuracy'])

model.fit(X_train, Y_train, batch_size=batch_size, 
          nb_epoch=nb_epoch,verbose=1,
          validation_data=(X_test, Y_test))

Evaluating the model on the test data

The code below implements this step in the deep learning workflow.

In [ ]:
#Evaluating the model on the test data    
score, accuracy = model.evaluate(X_test, Y_test, verbose=0)
print('Test score:', score)
print('Test accuracy:', accuracy)

# Adding more Convolution Layers

**Softmax Function:** Converts a vector of raw scores (logits) into a **probability distribution** — all values between 0 and 1 that sum to 1:

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

Softmax amplifies the largest values and suppresses smaller ones. It's used as the final layer in multi-class classification: the output tells you the model's confidence for each class.

**ReLU (Rectified Linear Unit):** The most common activation function in modern neural networks:

$$\text{ReLU}(x) = \max(0, x)$$

It simply passes positive values through and zeroes out negatives. Why it works so well:
- **No vanishing gradient** for positive inputs (gradient is always 1)
- **Sparse activation**: many neurons output 0, making the network efficient
- **Computationally fast**: just a threshold, no exponentials

Variants like Leaky ReLU ($\max(0.01x, x)$) and GELU fix the "dying ReLU" problem where neurons can permanently output 0.

In [ ]:
model = Sequential()
model.add(Convolution2D(nb_filters, nb_conv, nb_conv,
                        border_mode='valid',
                        input_shape=(1, img_rows, img_cols)))
model.add(Activation('relu'))
model.add(Convolution2D(nb_filters, nb_conv, nb_conv))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(nb_pool, nb_pool)))
model.add(Dropout(0.25))
    
model.add(Flatten())
model.add(Dense(128))
model.add(Activation('relu'))
model.add(Dropout(0.5))
model.add(Dense(nb_classes))
model.add(Activation('softmax'))

**Model Training (.fit()):** The `.fit()` method is where the model learns from data. It adjusts the model's internal parameters to minimize prediction errors on the training data.

For different model types, `.fit()` does different things:
- **Linear models**: Finds the best-fit line/plane (minimizes squared error)
- **Decision trees**: Recursively splits data to separate classes/values
- **Neural networks**: Runs gradient descent over many epochs
- **Transformers (StandardScaler, PCA)**: Computes statistics (mean, variance, components) from the training data

In [ ]:
model.compile(loss='categorical_crossentropy',
              optimizer='sgd',
              metrics=['accuracy'])

model.fit(X_train, Y_train, batch_size=batch_size, 
          nb_epoch=nb_epoch,verbose=1,
          validation_data=(X_test, Y_test))

Evaluating the model on the test data

The code below implements this step in the deep learning workflow.

In [ ]:
#Evaluating the model on the test data    
score, accuracy = model.evaluate(X_test, Y_test, verbose=0)
print('Test score:', score)
print('Test accuracy:', accuracy)

# Exercise

The above code has been written as a function. 

Change some of the **hyperparameters** and see what happens. 

In [ ]:
# Function for constructing the convolution neural network
# Feel free to add parameters, if you want

def build_model():
    """"""
    model = Sequential()
    model.add(Convolution2D(nb_filters, nb_conv, nb_conv,
                        border_mode='valid',
                        input_shape=(1, img_rows, img_cols)))
    model.add(Activation('relu'))
    model.add(Convolution2D(nb_filters, nb_conv, nb_conv))
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(nb_pool, nb_pool)))
    model.add(Dropout(0.25))
    
    model.add(Flatten())
    model.add(Dense(128))
    model.add(Activation('relu'))
    model.add(Dropout(0.5))
    model.add(Dense(nb_classes))
    model.add(Activation('softmax'))
    
    model.compile(loss='categorical_crossentropy',
              optimizer='sgd',
              metrics=['accuracy'])

    model.fit(X_train, Y_train, batch_size=batch_size, 
              nb_epoch=nb_epoch,verbose=1,
              validation_data=(X_test, Y_test))
          

    #Evaluating the model on the test data    
    score, accuracy = model.evaluate(X_test, Y_test, verbose=0)
    print('Test score:', score)
    print('Test accuracy:', accuracy)

Timing how long it takes to build the model and test it.

In [ ]:
#Timing how long it takes to build the model and test it.
%timeit -n1 -r1 build_model()

# Batch Normalisation

Normalize the activations of the previous layer at each batch, i.e. applies a transformation that maintains the mean activation close to 0 and the activation standard deviation close to 1.

## How to BatchNorm in Keras

```python
from keras.layers.normalization import BatchNormalization

BatchNormalization(epsilon=1e-06, mode=0, 
                   axis=-1, momentum=0.99, 
                   weights=None, beta_init='zero', 
                   gamma_init='one')
```

In [ ]:
# Try to add a new BatchNormalization layer to the Model 
# (after the Dropout layer)